# Étape 8 — RQ2 : quelles erreurs du graphe font perdre son bénéfice ?

L'étape 7 a confirmé que le graphe expert aide sur les **premières tentatives** quand les données sont rares (5 graines sur 5).
On abîme maintenant ce graphe de façon contrôlée et on mesure **combien de ce bénéfice il reste**.

- **Erreurs** : E1 fausses flèches aléatoires, E2 fausses flèches plausibles (même thème), E3 inversions, E4 suppressions, E5 cycles.
- **Taux** : 10 %, 30 %, 50 % du nombre de flèches expertes.
- **5 graines**, mêmes élèves que l'étape 7 pour une graine donnée : les références (expert, aléatoire, aucun) sont **reprises** de l'étape 7.
- **Ordre des blocs** : d'abord PrereqKT à 5 % (le résultat principal), puis PrereqKT à 1 %, puis GraphKT. Vous pouvez arrêter après le premier bloc et l'analyser.

**Mesure clé — bénéfice conservé :**
(AUC du graphe abîmé − AUC sans graphe) / (AUC du graphe expert − AUC sans graphe).
100 % = aussi bon que l'expert ; 0 % = aussi mauvais que pas de graphe du tout.

**GPU T4 obligatoire.** Environ 1 h par bloc. Reprise automatique en cas de déconnexion.

In [ ]:
# Cellule 1 — Drive, GPU, paramètres
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import stats

DOSSIER   = '/content/drive/MyDrive/projet_kt'
PREP      = os.path.join(DOSSIER, 'junyi_prepare')
RESULTATS = os.path.join(DOSSIER, 'resultats')
os.makedirs(RESULTATS, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Appareil :', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(pas de GPU : ce sera lent)')

HP = dict(dim=64, hidden=128, couches_graphe=2, dropout=0.2, lr=1e-3, batch=64, epoques_max=60, patience=5)
GRAINE = 42

In [ ]:
# Cellule 2 — Séquences, graphe expert, et repérage des PREMIÈRES tentatives
exos = pd.read_csv(os.path.join(PREP, 'exercices.csv'))
N = len(exos)
A_expert = np.load(os.path.join(PREP, 'adjacence_expert.npy')).astype(np.float32)
aretes_expert = list(zip(*np.nonzero(A_expert)))

def charger(partie):
    df = pd.read_csv(os.path.join(PREP, f'sequences_{partie}.csv')).sort_values(['uid', 'morceau'])
    U, Q, R, F = [], [], [], []
    vus = {}
    for uid, ex, rep in zip(df['uid'], df['exercices'], df['reponses']):
        q = np.array(ex.split(','), dtype=np.int64)
        r = np.array(rep.split(','), dtype=np.int64)
        deja = vus.setdefault(uid, set())          # exercices déjà tentés par cet élève (tous morceaux confondus)
        f = np.zeros(len(q), dtype=bool)
        for i, x in enumerate(q):
            if x not in deja:
                f[i] = True
                deja.add(x)
        U.append(uid); Q.append(q); R.append(r); F.append(f)
    return np.array(U), Q, R, F

donnees = {p: charger(p) for p in ['train', 'valid', 'test']}
for p, (u, q, r, f) in donnees.items():
    n_rep = sum(map(len, q)); n_first = sum(int(x.sum()) for x in f)
    print(f'{p:5s} : {len(set(u)):6,} élèves | {n_rep:9,} réponses | premières tentatives : {n_first:8,} ({n_first/n_rep:.1%})')

In [ ]:
# Cellule 3 — Tenseurs (Q exercices, R réponses, M masque, F première tentative)
def tenseurs(q_list, r_list, f_list):
    L = max(map(len, q_list))
    Q = np.zeros((len(q_list), L), dtype=np.int64); R = np.zeros_like(Q)
    M = np.zeros((len(q_list), L), dtype=bool);     F = np.zeros_like(M)
    for i, (q, r, f) in enumerate(zip(q_list, r_list, f_list)):
        Q[i, :len(q)], R[i, :len(r)], M[i, :len(q)], F[i, :len(f)] = q, r, True, f
    return tuple(torch.from_numpy(x) for x in (Q, R, M, F))

def sous_ensemble_train(fraction, graine=GRAINE):
    u, q, r, f = donnees['train']
    if fraction >= 1:
        return q, r, f
    rng = np.random.default_rng(graine)
    eleves = np.unique(u)
    gardes = set(rng.choice(eleves, size=max(1, int(fraction * len(eleves))), replace=False))
    idx = [i for i, x in enumerate(u) if x in gardes]
    return [q[i] for i in idx], [r[i] for i in idx], [f[i] for i in idx]

T_valid = tenseurs(*donnees['valid'][1:])
T_test  = tenseurs(*donnees['test'][1:])
print('OK')

### Les modèles

- **GraphKT** : identique à l'étape 5 (le graphe lisse les représentations des exercices, avec un canal « prérequis » et un canal « successeurs »).
- **PrereqKT** : GraphKT **plus** 6 indicateurs calculés à chaque instant pour l'exercice à prédire :
  taux de réussite moyen de l'élève sur ses **prérequis** déjà tentés, part des prérequis déjà tentés, présence de prérequis, et les trois mêmes indicateurs pour ses **successeurs**.
  C'est la traduction directe de l'hypothèse « *s'il a échoué sur A, il échouera plus souvent sur B* ».
  Avec un graphe vide, ces indicateurs valent 0 : PrereqKT redevient GraphKT sans graphe.

In [ ]:
# Cellule 4 — Modèles
def normaliser_lignes(A):
    deg = A.sum(1, keepdims=True)
    return np.divide(A, deg, out=np.zeros_like(A), where=deg > 0)

class GraphKT(nn.Module):
    def __init__(self, n, hp, A, indicateurs=False):
        super().__init__()
        d, h = hp['dim'], hp['hidden']
        A = np.asarray(A, dtype=np.float32)
        self.register_buffer('P_pre', torch.from_numpy(normaliser_lignes(A.T.copy())))  # ligne j : prérequis de j
        self.register_buffer('P_suc', torch.from_numpy(normaliser_lignes(A.copy())))    # ligne i : successeurs de i
        self.n, self.indicateurs = n, indicateurs
        self.E = nn.Embedding(n, d)
        self.couches = nn.ModuleList([nn.ModuleDict({'pre': nn.Linear(d, d), 'suc': nn.Linear(d, d)})
                                      for _ in range(hp['couches_graphe'])])
        self.R = nn.Embedding(2, d)
        self.lstm = nn.LSTM(2 * d, h, batch_first=True)
        self.drop = nn.Dropout(hp['dropout'])
        n_ind = 6 if indicateurs else 0
        self.pred = nn.Sequential(nn.Linear(h + d + n_ind, h), nn.ReLU(), nn.Linear(h, 1))

    def representations(self):
        H = self.E.weight
        a_pre = (self.P_pre.sum(1, keepdim=True) > 0).float()
        a_suc = (self.P_suc.sum(1, keepdim=True) > 0).float()
        for c in self.couches:
            H = H + torch.relu(c['pre'](self.P_pre @ H)) * a_pre + torch.relu(c['suc'](self.P_suc @ H)) * a_suc
        return H

    def calcul_indicateurs(self, q, r):
        # Historique cumulé par exercice jusqu'à l'instant t : nombre d'essais et de réussites (B, L-1, N)
        B, L = q.shape
        essais = torch.zeros(B, L - 1, self.n, device=q.device)
        essais.scatter_(2, q[:, :-1].unsqueeze(-1), 1.0)
        reussites = essais * r[:, :-1].unsqueeze(-1).float()
        essais, reussites = essais.cumsum(1), reussites.cumsum(1)
        tente = (essais > 0).float()
        taux = reussites / essais.clamp(min=1)
        sortie = []
        for P in (self.P_pre, self.P_suc):
            voisins = (P[q[:, 1:]] > 0).float()                 # voisins de l'exercice à prédire (B, L-1, N)
            n_vois = voisins.sum(-1)
            n_tentes = (voisins * tente).sum(-1)
            moy = (voisins * tente * taux).sum(-1) / n_tentes.clamp(min=1)
            sortie += [moy, n_tentes / n_vois.clamp(min=1), (n_vois > 0).float()]
        return torch.stack(sortie, -1)

    def forward(self, q, r):
        H = self.representations()
        h, _ = self.lstm(torch.cat([H[q[:, :-1]], self.R(r[:, :-1])], dim=-1))
        x = [self.drop(h), H[q[:, 1:]]]
        if self.indicateurs:
            x.append(self.calcul_indicateurs(q, r))
        return self.pred(torch.cat(x, dim=-1)).squeeze(-1)

print('OK')

In [ ]:
# Cellule 5 — Graphes : expert, aucun, aléatoire, et corruption contrôlée E1–E5
import networkx as nx

def vers_matrice(aretes, n=N):
    A = np.zeros((n, n), dtype=np.float32)
    for a, b in aretes:
        A[a, b] = 1
    return A

def corrompre(aretes, type_err, taux, graine=GRAINE, topics=exos['topic'].values):
    rng = random.Random(10_000 + graine)
    E = set(map(tuple, aretes))
    k = int(round(taux * len(E)))
    noeuds = list(range(N))
    if type_err == 'E1':      # fausses flèches aléatoires
        while k > 0:
            a, b = rng.sample(noeuds, 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E2':    # fausses flèches plausibles : même thème
        par_topic = {}
        for i, t in enumerate(topics):
            par_topic.setdefault(t, []).append(i)
        groupes = [g for g in par_topic.values() if len(g) >= 2]
        essais = 0
        while k > 0 and essais < 100 * len(E) + 1000:
            essais += 1
            a, b = rng.sample(rng.choice(groupes), 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E3':    # inversions
        for a, b in rng.sample(sorted(E), min(k, len(E))):
            E.discard((a, b)); E.add((b, a))
    elif type_err == 'E4':    # suppressions
        for e in rng.sample(sorted(E), min(k, len(E))):
            E.discard(e)
    elif type_err == 'E5':    # cycles : C -> A alors que A mène déjà à C
        G = nx.DiGraph(list(E))
        candidats = sorted(E); rng.shuffle(candidats)
        for a, b in candidats:
            if k == 0:
                break
            c = rng.choice(list(nx.descendants(G, b)) + [b])
            if c != a and (c, a) not in E:
                E.add((c, a)); G.add_edge(c, a); k -= 1
    return sorted(E)

def graphe(nom, graine=GRAINE, taux=0.0):
    if nom == 'expert':
        return A_expert
    if nom == 'aucun':
        return np.zeros((N, N), dtype=np.float32)
    if nom == 'aleatoire':
        rng = random.Random(graine)
        E = set()
        while len(E) < len(aretes_expert):
            a, b = rng.sample(range(N), 2)
            E.add((a, b))
        return vers_matrice(E)
    return vers_matrice(corrompre(aretes_expert, nom, taux, graine))

def f1_aretes(A, B=A_expert):
    P, V = set(zip(*np.nonzero(A))), set(zip(*np.nonzero(B)))
    tp = len(P & V); p = tp / max(1, len(P)); r = tp / max(1, len(V))
    return round(2 * p * r / (p + r), 3) if p + r else 0.0

print(f'{"graphe":8s} {"taux":>5s} {"flèches":>8s} {"F1/expert":>10s} {"acyclique":>10s}')
for t in ['E1', 'E2', 'E3', 'E4', 'E5']:
    for taux in [0.1, 0.3, 0.5]:
        A = graphe(t, 0, taux)
        print(f'{t:8s} {taux:5.0%} {int(A.sum()):8d} {f1_aretes(A):10.3f} {str(nx.is_directed_acyclic_graph(nx.DiGraph(A))):>10s}')

In [ ]:
# Cellule 6 — Entraînement et évaluation par sous-ensemble
def fixer_graine(g):
    random.seed(g); np.random.seed(g); torch.manual_seed(g); torch.cuda.manual_seed_all(g)

@torch.no_grad()
def evaluer(modele, T, batch=128):
    modele.eval()
    Q, R, M, F = T
    ys, ps, fs = [], [], []
    for i in range(0, len(Q), batch):
        q, r, m, f = (x[i:i+batch].to(DEVICE) for x in (Q, R, M, F))
        p = torch.sigmoid(modele(q, r))
        c = m[:, 1:]
        ys.append(r[:, 1:][c].cpu().numpy()); ps.append(p[c].float().cpu().numpy()); fs.append(f[:, 1:][c].cpu().numpy())
    y, p, f = (np.concatenate(x) for x in (ys, ps, fs))
    auc = lambda s: round(float(roc_auc_score(y[s], p[s])), 4) if len(set(y[s])) == 2 else None
    tout = np.ones_like(f, dtype=bool)
    return {'AUC': auc(tout), 'AUC_1re': auc(f), 'AUC_rep': auc(~f), 'ACC': round(float(accuracy_score(y, p >= 0.5)), 4)}

def entrainer(nom_modele, A, fraction=1.0, graine=GRAINE, hp=HP):
    fixer_graine(graine)
    Q, R, M, F = tenseurs(*sous_ensemble_train(fraction, graine))
    modele = GraphKT(N, hp, A, indicateurs=(nom_modele == 'PrereqKT')).to(DEVICE)
    opt = torch.optim.Adam(modele.parameters(), lr=hp['lr'])
    perte = nn.BCEWithLogitsLoss()
    meilleur, etat, attente, t0 = -1, None, 0, time.time()
    for ep in range(hp['epoques_max']):
        modele.train()
        ordre = torch.randperm(len(Q))
        for i in range(0, len(Q), hp['batch']):
            idx = ordre[i:i + hp['batch']]
            q, r, m = Q[idx].to(DEVICE), R[idx].to(DEVICE), M[idx].to(DEVICE)
            c = m[:, 1:]
            l = perte(modele(q, r)[c], r[:, 1:][c].float())
            opt.zero_grad(); l.backward()
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0)
            opt.step()
        v = evaluer(modele, T_valid)['AUC']
        print(f'   époque {ep+1:2d} | AUC valid {v:.4f} | {(time.time()-t0)/60:.1f} min')
        if v > meilleur + 1e-4:
            meilleur, etat, attente = v, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= hp['patience']:
                break
    modele.load_state_dict(etat)
    return {'AUC_valid': round(meilleur, 4), **{f'{k}_test': v for k, v in evaluer(modele, T_test).items()},
            'epoques': ep + 1, 'minutes': round((time.time() - t0) / 60, 1)}
print('OK')

In [ ]:
# Cellule 7 — Grille RQ2 (reprise automatique ; les références viennent de l'étape 7)
BLOCS = [('PrereqKT', 0.05), ('PrereqKT', 0.01), ('GraphKT', 0.05), ('GraphKT', 0.01)]
ERREURS = ['E1', 'E2', 'E3', 'E4', 'E5']
TAUX    = [0.1, 0.3, 0.5]
GRAINES = [0, 1, 2, 3, 4]

# Références (taux 0) reprises de l'étape 7
ref = pd.read_csv(os.path.join(RESULTATS, 'phase2_confirmation.csv'))
ref['taux'] = 0.0

chemin_res = os.path.join(RESULTATS, 'rq2_grille.csv')
resultats = pd.read_csv(chemin_res).to_dict('records') if os.path.exists(chemin_res) else []
fait = {(x['modele'], x['graphe'], float(x['fraction']), float(x['taux']), int(x['graine'])) for x in resultats}
t_debut = time.time()

for nom, frac in BLOCS:
    a_faire = [(e, t, g) for e in ERREURS for t in TAUX for g in GRAINES
               if (nom, e, float(frac), float(t), g) not in fait]
    print(f'\n=========== Bloc {nom} à {frac:.0%} : {len(a_faire)} entraînements restants ===========')
    for i, (e, t, g_) in enumerate(a_faire, 1):
        res = entrainer(nom, graphe(e, graine=g_, taux=t), fraction=frac, graine=g_)
        resultats.append({'modele': nom, 'graphe': e, 'taux': t, 'fraction': frac, 'graine': g_, **res})
        pd.DataFrame(resultats).to_csv(chemin_res, index=False)
        print(f'[{i}/{len(a_faire)}] {e} {t:.0%} graine {g_} -> AUC 1res {res["AUC_1re_test"]} '
              f'({res["epoques"]} ép., écoulé {(time.time()-t_debut)/60:.0f} min)')

In [ ]:
# Cellule 8 — Analyse, figure et résumé à copier dans la conversation
import matplotlib.pyplot as plt

grille = pd.concat([ref, pd.DataFrame(resultats)], ignore_index=True)
grille = grille[grille.modele.isin({m for m, _ in BLOCS})]
col = 'AUC_1re_test'

lignes = []
for (nom, frac), s in grille.groupby(['modele', 'fraction']):
    if not (s.graphe.isin(ERREURS)).any():
        continue
    base = s[s.graphe.isin(['expert', 'aucun', 'aleatoire'])].pivot_table(index='graine', columns='graphe', values=col)
    for e in ERREURS:
        for t in TAUX:
            c = s[(s.graphe == e) & (s.taux == t)].set_index('graine')[col]
            gr = c.index.intersection(base.index)
            if len(gr) < 2:
                continue
            gain_exp = base.loc[gr, 'expert'] - base.loc[gr, 'aucun']
            conserve = ((c.loc[gr] - base.loc[gr, 'aucun']) / gain_exp).mean()
            d = c.loc[gr] - base.loc[gr, 'expert']
            p = stats.wilcoxon(d, alternative='less').pvalue if (d != 0).any() else 1.0
            lignes.append({'modele': nom, 'fraction': frac, 'erreur': e, 'taux': t, 'n': len(gr),
                           'AUC_1re': round(c.loc[gr].mean(), 4), 'sd': round(c.loc[gr].std(), 4),
                           'ecart_vs_expert': round(d.mean(), 4), 'graines_pires_que_expert': f'{int((d < 0).sum())}/{len(gr)}',
                           'p_wilcoxon': round(p, 4), 'benefice_conserve_%': round(100 * conserve, 1),
                           'mieux_que_aleatoire': f'{int((c.loc[gr] > base.loc[gr, "aleatoire"]).sum())}/{len(gr)}'})
synthese = pd.DataFrame(lignes)
synthese.to_csv(os.path.join(RESULTATS, 'rq2_synthese.csv'), index=False)

for (nom, frac), s in synthese.groupby(['modele', 'fraction']):
    b = grille[(grille.modele == nom) & (grille.fraction == frac)]
    refs = {g: b[b.graphe == g][col].mean() for g in ['expert', 'aleatoire', 'aucun']}
    print(f'\n===== {nom} | {frac:.0%} des élèves | expert {refs["expert"]:.4f} · aléatoire {refs["aleatoire"]:.4f} · aucun {refs["aucun"]:.4f} =====')
    print(s.drop(columns=['modele', 'fraction']).to_string(index=False))
    print('Bénéfice conservé (%) :')
    print(s.pivot_table(index='erreur', columns='taux', values='benefice_conserve_%').to_string())

    # Figure : AUC des premières tentatives selon le taux d'erreur, une courbe par type
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    for e in ERREURS:
        pts = [(0.0, refs['expert'])] + [(t, s[(s.erreur == e) & (s.taux == t)]['AUC_1re'].mean()) for t in TAUX]
        ax.plot([100 * x for x, _ in pts], [y for _, y in pts], marker='o', label=e)
    ax.axhline(refs['aleatoire'], ls='--', c='grey', lw=1, label='graphe aléatoire')
    ax.axhline(refs['aucun'], ls=':', c='black', lw=1, label='aucun graphe')
    ax.set_xlabel("Taux d'erreur injecté (%)"); ax.set_ylabel('AUC (premières tentatives)')
    ax.set_title(f'{nom} — {frac:.0%} des élèves'); ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTATS, f'rq2_{nom}_{int(frac*100)}pc.png'), dpi=200)
    plt.show()